# Spike Detection

This notebook trains and tests the SmogNet spike detector using the same reusable project code as the dashboard pipeline.

Important note about accuracy: the supplied data does not include a manually labeled `is_spike` target. Because of that, the notebook reports **proxy accuracy** against a transparent validation label: rows with `main_aqi >= 4` are treated as high-pollution events. This is useful for model checking, but it should be described as validation against an AQI proxy, not a confirmed ground-truth accuracy score.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.anomaly_detection import (
    compute_rolling_baselines,
    detect_spikes,
    learn_anomaly_thresholds,
    score_test_data,
)
from src.config import DEFAULT_ROLLING_WINDOW, RAW_DATA_DIR
from src.data_loader import load_air_quality_data
from src.preprocessing import detect_pollutant_columns, preprocess_air_quality_data

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PROJECT_ROOT

PosixPath('/Users/waleedhassan/Downloads/Datathon/Somgnet-V1.1.1')

## 1. Load and Preprocess Data

The training files are used only to learn rolling baselines and score thresholds. The testing files are held out for spike detection and validation.

In [2]:
raw_train, raw_test, split_metadata = load_air_quality_data(RAW_DATA_DIR)

train_df = preprocess_air_quality_data(raw_train, is_train=True)
test_df = preprocess_air_quality_data(raw_test, is_train=False)

pollutant_cols = [
    pollutant
    for pollutant in detect_pollutant_columns(train_df)
    if pollutant in test_df.columns
]

print(f"Split strategy: {split_metadata['split_strategy']}")
print(f"Training rows: {len(train_df):,}")
print(f"Testing rows: {len(test_df):,}")
print(f"Pollutants used: {pollutant_cols}")

train_df.head()

[load] Warning: trimming 18305 training row(s) at or after the test start 2024-07-01 00:00:00 to prevent temporal leakage.
Split strategy: filename
Training rows: 122,896
Testing rows: 21,792
Pollutants used: ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']


,datetime,main_aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,shortwave_radiation,city,timestamp,hour,day,month,year,date,season
0,2021-08-24 00:00:00,5,"1,228.330",0.000,27.760,40.410,6.020,66.960,87.070,14.690,29.700,55.000,19.700,0.000,943.400,10.500,74.000,0,Islamabad,2021-08-24 00:00:00,0,24,8,2021,2021-08-24,Monsoon/Summer
1,2021-08-24 01:00:00,5,"1,134.870",0.000,24.330,46.490,6.200,64.500,82.370,14.570,29.400,56.000,19.700,0.000,943.200,10.800,92.000,0,Islamabad,2021-08-24 01:00:00,1,24,8,2021,2021-08-24,Monsoon/Summer
2,2021-08-24 02:00:00,5,"1,361.850",0.940,39.410,30.760,6.910,64.210,80.380,16.210,28.900,58.000,19.800,0.000,943.000,10.100,107.000,0,Islamabad,2021-08-24 02:00:00,2,24,8,2021,2021-08-24,Monsoon/Summer
3,2021-08-24 03:00:00,5,"1,682.280",8.380,51.410,27.180,9.060,64.750,79.550,17.730,28.400,60.000,19.800,0.000,942.800,10.500,106.000,0,Islamabad,2021-08-24 03:00:00,3,24,8,2021,2021-08-24,Monsoon/Summer
4,2021-08-24 04:00:00,5,"1,054.760",3.070,27.080,91.550,21.700,59.860,71.180,13.300,28.100,62.000,20.100,0.000,942.900,9.400,97.000,0,Islamabad,2021-08-24 04:00:00,4,24,8,2021,2021-08-24,Monsoon/Summer


## 2. Train the Spike Detector

Training means learning city/season rolling medians, rolling IQR baselines, and adaptive quantile thresholds from the historical training split.

In [3]:
train_scored, baseline_reference = compute_rolling_baselines(
    train_df,
    pollutant_cols,
    window=DEFAULT_ROLLING_WINDOW,
)
thresholds = learn_anomaly_thresholds(train_scored, pollutant_cols)

threshold_preview = pd.concat(
    [table.assign(threshold_level=level) for level, table in thresholds["tables"].items()],
    ignore_index=True,
)

threshold_preview.head(10)

,city,season,pollutant,moderate_threshold,high_threshold,severe_threshold,n_obs,level,threshold_level
0,Islamabad,Monsoon/Summer,co,2.935,4.509,7.275,7368,city_season,city_season
1,Islamabad,Monsoon/Summer,no,7.875,17.834,48.425,7368,city_season,city_season
2,Islamabad,Monsoon/Summer,no2,2.185,3.182,5.548,7368,city_season,city_season
3,Islamabad,Monsoon/Summer,o3,1.389,1.647,2.095,7368,city_season,city_season
4,Islamabad,Monsoon/Summer,so2,1.986,2.653,3.290,7368,city_season,city_season
5,Islamabad,Monsoon/Summer,pm2_5,2.207,3.164,4.331,7368,city_season,city_season
6,Islamabad,Monsoon/Summer,pm10,2.226,3.071,4.392,7368,city_season,city_season
7,Islamabad,Monsoon/Summer,nh3,3.674,6.251,10.350,7368,city_season,city_season
8,Islamabad,Autumn,co,2.182,2.762,3.523,4367,city_season,city_season
9,Islamabad,Autumn,no,2.720,3.888,5.658,4367,city_season,city_season


## 3. Test the Spike Detector

The test data is scored against training-derived references only, then rows crossing learned thresholds are flagged as spikes.

In [4]:
test_scored = score_test_data(test_df, baseline_reference, pollutant_cols)
test_scored = test_scored.copy()
test_scored["row_id"] = np.arange(len(test_scored))

detected_spikes = detect_spikes(test_scored, thresholds, pollutant_cols)
test_eval = test_scored.copy()
test_eval["predicted_spike"] = test_eval["row_id"].isin(detected_spikes["row_id"])

print(f"Detected spikes: {len(detected_spikes):,}")
print(f"Spike rate in test data: {test_eval['predicted_spike'].mean():.2%}")

detected_spikes[[
    "timestamp",
    "city",
    "dominant_pollutant",
    "combined_anomaly_score",
    "severity",
    "anomaly_explanation",
]].head(10)

Detected spikes: 3,763
Spike rate in test data: 17.27%


,timestamp,city,dominant_pollutant,combined_anomaly_score,severity,anomaly_explanation
0,2024-07-01 00:00:00,Karachi,pm2_5,3.476,high,PM2_5 is 3.48 robust-IQR units above its city ...
1,2024-07-01 01:00:00,Karachi,pm2_5,4.385,high,PM2_5 is 4.39 robust-IQR units above its city ...
2,2024-07-01 02:00:00,Karachi,pm2_5,4.753,high,PM2_5 is 4.75 robust-IQR units above its city ...
3,2024-07-01 03:00:00,Karachi,pm2_5,4.196,high,PM2_5 is 4.20 robust-IQR units above its city ...
4,2024-07-01 04:00:00,Karachi,pm2_5,2.945,moderate,PM2_5 is 2.95 robust-IQR units above its city ...
5,2024-07-01 06:00:00,Quetta,o3,1.953,severe,O3 is 1.95 robust-IQR units above its city sea...
6,2024-07-01 07:00:00,Quetta,o3,1.231,moderate,O3 is 1.23 robust-IQR units above its city sea...
7,2024-07-01 08:00:00,Karachi,pm2_5,3.167,moderate,PM2_5 is 3.17 robust-IQR units above its city ...
8,2024-07-01 09:00:00,Karachi,pm2_5,3.784,high,PM2_5 is 3.78 robust-IQR units above its city ...
9,2024-07-01 10:00:00,Karachi,pm2_5,3.827,high,PM2_5 is 3.83 robust-IQR units above its city ...


## 4. Check Accuracy with an AQI Proxy

`main_aqi >= 4` is used as a proxy for a meaningful pollution event. This checks whether detected spikes align with high-AQI periods, while keeping clear that the competition data does not provide true spike labels.

In [5]:
if "main_aqi" not in test_eval.columns:
    raise ValueError("The AQI proxy metric requires a main_aqi column in the test data.")

test_eval["proxy_high_pollution_event"] = pd.to_numeric(
    test_eval["main_aqi"], errors="coerce"
).fillna(0).ge(4)

y_true = test_eval["proxy_high_pollution_event"]
y_pred = test_eval["predicted_spike"]

spike_metrics = pd.DataFrame(
    [
        {
            "proxy_accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1_score": f1_score(y_true, y_pred, zero_division=0),
            "proxy_positive_rate": y_true.mean(),
            "predicted_spike_rate": y_pred.mean(),
        }
    ]
)

spike_metrics

,proxy_accuracy,precision,recall,f1_score,proxy_positive_rate,predicted_spike_rate
0,0.460,0.817,0.217,0.343,0.649,0.173


In [6]:
cm = confusion_matrix(y_true, y_pred, labels=[False, True])
cm_df = pd.DataFrame(
    cm,
    index=["Proxy normal", "Proxy high pollution"],
    columns=["Predicted normal", "Predicted spike"],
)

display(cm_df)

report = classification_report(
    y_true,
    y_pred,
    labels=[False, True],
    target_names=["Proxy normal", "Proxy high pollution"],
    zero_division=0,
    output_dict=True,
)
pd.DataFrame(report).T

,Predicted normal,Predicted spike
Proxy normal,6952,689
Proxy high pollution,11077,3074


,precision,recall,f1-score,support
Proxy normal,0.386,0.910,0.542,"7,641.000"
Proxy high pollution,0.817,0.217,0.343,"14,151.000"
accuracy,0.460,0.460,0.460,0.460
macro avg,0.601,0.564,0.442,"21,792.000"
weighted avg,0.666,0.460,0.413,"21,792.000"


## 5. Inspect Model Behavior

These summaries show which cities, pollutants, and severities are driving the spike detections.

In [7]:
severity_by_city = (
    detected_spikes.groupby(["city", "severity"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["moderate", "high", "severe"], fill_value=0)
)

dominant_pollutants = detected_spikes["dominant_pollutant"].value_counts().rename_axis("pollutant").to_frame("spike_count")

display(severity_by_city)
display(dominant_pollutants)

severity,moderate,high,severe
city,,,
Islamabad,259,152,87
Karachi,550,400,225
Lahore,397,259,97
Peshawar,248,142,130
Quetta,504,257,56


,spike_count
pollutant,
so2,668
pm2_5,658
o3,613
no2,501
no,471
pm10,454
nh3,358
co,40


In [10]:
# fig = px.scatter(
#     detected_spikes,
#     x="timestamp",
#     y="city",
#     color="severity",
#     size="combined_anomaly_score",
#     hover_data=["dominant_pollutant", "combined_anomaly_score"],
#     category_orders={"severity": ["moderate", "high", "severe"]},
#     title="Detected Spikes Across the Test Period",
# )
# fig.update_layout(template="plotly_white")
# fig.show()
from IPython.display import HTML, display
from plotly.basedatatypes import BaseFigure

def show_plot(fig, *args, **kwargs):
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

BaseFigure.show = show_plot

